# 🫁 DenseNet-121 — NIH ChestX-ray14
### 4 Classes : Pneumonia | Effusion | Cardiomegaly | Normal
**PFE — Licence IA 2024-2025**

---
**Instructions:**
1. `Runtime` → `Change runtime type` → **GPU (T4)**
2. Run all cells in order (Ctrl+F9)
3. Results saved to Google Drive automatically


## ✅ Cell 1 — Check GPU

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    !nvidia-smi
else:
    print('⚠️  No GPU found — training will be slow.')
    print('   Go to Runtime → Change runtime type → GPU')

## 📦 Cell 2 — Install dependencies

In [ ]:
%%capture
!pip install -q torch torchvision
!pip install -q scikit-learn matplotlib seaborn opencv-python-headless tqdm pandas Pillow
print('✅ All packages ready')

## 💾 Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Folder structure in your Drive ──────────────────
# MyDrive/
#   PFE_ChestXray/
#     images/          ← all NIH .png images here
#     Data_Entry_2017.csv
#     checkpoints/     ← best model saved here
#     logs/            ← plots saved here

BASE_DIR  = '/content/drive/MyDrive/PFE_ChestXray'
DATA_DIR  = os.path.join(BASE_DIR, 'images')
CSV_PATH  = os.path.join(BASE_DIR, 'Data_Entry_2017.csv')
SAVE_DIR  = os.path.join(BASE_DIR, 'checkpoints')
LOG_DIR   = os.path.join(BASE_DIR, 'logs')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

print(f'Base    : {BASE_DIR}')
print(f'Images  : {DATA_DIR}')
print(f'CSV     : {CSV_PATH}')
print(f'Saves   : {SAVE_DIR}')
print(f'Logs    : {LOG_DIR}')

# Quick sanity check
if os.path.exists(CSV_PATH):
    print('✅ CSV found')
else:
    print('❌ CSV not found — check your Drive path')

if os.path.exists(DATA_DIR):
    n_imgs = len([f for f in os.listdir(DATA_DIR) if f.endswith('.png')])
    print(f'✅ Images folder found — {n_imgs} .png files')
else:
    print('❌ Images folder not found — check your Drive path')

## ⚙️ Cell 4 — Configuration

In [ ]:
import random
import numpy as np

# ── Tweak these if needed ───────────────────────────
CLASSES      = ['Pneumonia', 'Effusion', 'Cardiomegaly', 'No Finding']
CLASS_NAMES  = ['Pneumonia', 'Effusion', 'Cardiomegaly', 'Normal']
NUM_CLASSES  = 4

IMAGE_SIZE   = 224
BATCH_SIZE   = 32 if device == 'cuda' else 16   # smaller on CPU
NUM_EPOCHS_P1 = 15   # Phase 1: frozen backbone
NUM_EPOCHS_P2 = 15   # Phase 2: full fine-tune
LR           = 1e-4
WEIGHT_DECAY = 1e-5
EARLY_STOP   = 7     # patience
SEED         = 42
NUM_WORKERS  = 2     # Colab recommended
USE_AMP      = (device == 'cuda')   # mixed precision on GPU only

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
# ────────────────────────────────────────────────────

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f'✅ Config ready | Device={device} | Batch={BATCH_SIZE} | AMP={USE_AMP}')

## 📊 Cell 5 — Load & Filter CSV

In [ ]:
import pandas as pd

def load_and_filter_csv(csv_path):
    df = pd.read_csv(csv_path)
    records = []
    for _, row in df.iterrows():
        img_labels = [l.strip() for l in row['Finding Labels'].split('|')]
        matched = None
        for idx, cls in enumerate(CLASSES):
            if cls in img_labels:
                matched = idx
                break
        if matched is None and 'No Finding' in img_labels:
            matched = 3
        if matched is not None:
            records.append({'Image Index': row['Image Index'], 'label': matched})

    filtered = pd.DataFrame(records)
    print(f'Total images kept: {len(filtered):,}')
    print('-' * 35)
    for i, name in enumerate(CLASS_NAMES):
        c = (filtered['label'] == i).sum()
        bar = '█' * int(c / len(filtered) * 40)
        print(f'  {name:15s} {c:6,}  {bar}')
    return filtered

df = load_and_filter_csv(CSV_PATH)

## ✂️ Cell 6 — Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=(1 - TRAIN_RATIO), stratify=df['label'], random_state=SEED
)
val_ratio_adj = VAL_RATIO / (1 - TRAIN_RATIO)
val_df, test_df = train_test_split(
    temp_df, test_size=(1 - val_ratio_adj), stratify=temp_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train  : {len(train_df):,} images')
print(f'Val    : {len(val_df):,} images')
print(f'Test   : {len(test_df):,} images')

## 🖼️ Cell 7 — Dataset & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from PIL import Image

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 20, IMAGE_SIZE + 20)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image Index'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

# Weighted sampler
labels_np  = train_df['label'].values
counts     = np.bincount(labels_np, minlength=NUM_CLASSES).astype(float)
weights    = 1.0 / counts
sample_w   = weights[labels_np]
sampler    = WeightedRandomSampler(torch.DoubleTensor(sample_w), len(sample_w), replacement=True)

train_ds = ChestXrayDataset(train_df, DATA_DIR, train_transform)
val_ds   = ChestXrayDataset(val_df,   DATA_DIR, val_test_transform)
test_ds  = ChestXrayDataset(test_df,  DATA_DIR, val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=(device=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=='cuda'))

print(f'✅ DataLoaders ready')
print(f'   Train batches : {len(train_loader)}')
print(f'   Val   batches : {len(val_loader)}')
print(f'   Test  batches : {len(test_loader)}')

## 🔍 Cell 8 — Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
shown = {0: 0, 1: 0, 2: 0, 3: 0}

for idx in range(len(train_df)):
    row   = train_df.iloc[idx]
    label = int(row['label'])
    if shown[label] >= 2:
        continue
    img = Image.open(os.path.join(DATA_DIR, row['Image Index'])).convert('L')
    ax  = axes[shown[label]][label]
    ax.imshow(img, cmap='bone')
    ax.set_title(CLASS_NAMES[label], fontsize=11)
    ax.axis('off')
    shown[label] += 1
    if all(v >= 2 for v in shown.values()):
        break

plt.suptitle('Sample X-rays per class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'sample_images.png'), dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sample grid saved')

## 🧠 Cell 9 — Build DenseNet-121 Model

In [ ]:
import torch.nn as nn
import torchvision.models as models

def build_model(num_classes=4, pretrained=True):
    model = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze last dense block + BN
    for name, param in model.named_parameters():
        if 'denseblock4' in name or 'norm5' in name:
            param.requires_grad = True

    # Replace classifier
    in_features = model.classifier.in_features  # 1024
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(512, num_classes)
    )
    return model

model = build_model(NUM_CLASSES).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ DenseNet-121 ready')
print(f'   Total params    : {total:,}')
print(f'   Trainable params: {trainable:,} ({100*trainable/total:.1f}%)')
print(f'   Classifier head : Linear(1024→512) → ReLU → Dropout → Linear(512→{NUM_CLASSES})')

## ⚖️ Cell 10 — Loss, Optimizer, Scheduler

In [ ]:
import torch.optim as optim

# Class weights (inverse frequency)
counts_arr = np.bincount(train_df['label'].values, minlength=NUM_CLASSES).astype(float)
cw         = 1.0 / (counts_arr / counts_arr.sum())
cw         = cw / cw.sum() * NUM_CLASSES
class_weights = torch.FloatTensor(cw).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

print('✅ Loss / Optimizer / Scheduler ready')
print(f'   Class weights : {[f"{w:.2f}" for w in cw]}')
print(f'   LR            : {LR}')
print(f'   Scheduler     : ReduceLROnPlateau (patience=3, factor=0.5)')

## 🔁 Cell 11 — Training & Validation Functions

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  Train', leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, desc='Val'):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    pbar = tqdm(loader, desc=f'  {desc}', leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
    try:
        auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
    except ValueError:
        auc = 0.0

    return total_loss / total, correct / total, auc, all_probs, all_labels

print('✅ Train / Eval functions ready')

## 🚀 Cell 12 — Phase 1 Training (Frozen Backbone)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc   = 0.0
no_improve = 0

print('━'*60)
print('  PHASE 1 — Classifier head only (backbone frozen)')
print('━'*60)
print(f'{"Epoch":>5} | {"T-Loss":>7} | {"T-Acc":>6} | {"V-Loss":>7} | {"V-Acc":>6} | {"V-AUC":>6}')
print('-'*50)

for epoch in range(1, NUM_EPOCHS_P1 + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    v_loss, v_acc, v_auc, _, _ = evaluate(model, val_loader, criterion)

    scheduler.step(v_auc)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    history['val_auc'].append(v_auc)

    flag = ''
    if v_auc > best_auc:
        best_auc   = v_auc
        no_improve = 0
        torch.save({
            'epoch': epoch, 'phase': 1,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_auc': best_auc,
            'classes': CLASS_NAMES,
        }, os.path.join(SAVE_DIR, 'best_densenet121.pth'))
        flag = '  ✓ saved'
    else:
        no_improve += 1

    print(f'{epoch:>5} | {t_loss:>7.4f} | {t_acc:>6.4f} | {v_loss:>7.4f} | {v_acc:>6.4f} | {v_auc:>6.4f}{flag}')

    if no_improve >= EARLY_STOP:
        print(f'\nEarly stop at epoch {epoch}')
        break

print(f'\n✅ Phase 1 done — Best AUC: {best_auc:.4f}')

## 🔓 Cell 13 — Phase 2 Fine-tuning (All Layers)

In [ ]:
# Unfreeze entire network with differential LR
for param in model.parameters():
    param.requires_grad = True

backbone_params   = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

optimizer2 = optim.Adam([
    {'params': backbone_params,   'lr': 1e-5},
    {'params': classifier_params, 'lr': 1e-4},
], weight_decay=WEIGHT_DECAY)

scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode='max', factor=0.5, patience=3, verbose=True
)

no_improve = 0

print('━'*60)
print('  PHASE 2 — Full fine-tuning (all layers unfrozen)')
print('  Backbone LR=1e-5 | Classifier LR=1e-4')
print('━'*60)
print(f'{"Epoch":>5} | {"T-Loss":>7} | {"T-Acc":>6} | {"V-Loss":>7} | {"V-Acc":>6} | {"V-AUC":>6}')
print('-'*50)

for epoch in range(1, NUM_EPOCHS_P2 + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer2)
    v_loss, v_acc, v_auc, _, _ = evaluate(model, val_loader, criterion)

    scheduler2.step(v_auc)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    history['val_auc'].append(v_auc)

    flag = ''
    if v_auc > best_auc:
        best_auc   = v_auc
        no_improve = 0
        torch.save({
            'epoch': epoch, 'phase': 2,
            'model_state': model.state_dict(),
            'optim_state': optimizer2.state_dict(),
            'val_auc': best_auc,
            'classes': CLASS_NAMES,
        }, os.path.join(SAVE_DIR, 'best_densenet121.pth'))
        flag = '  ✓ saved'
    else:
        no_improve += 1

    print(f'{epoch:>5} | {t_loss:>7.4f} | {t_acc:>6.4f} | {v_loss:>7.4f} | {v_acc:>6.4f} | {v_auc:>6.4f}{flag}')

    if no_improve >= EARLY_STOP:
        print(f'\nEarly stop at epoch {epoch}')
        break

print(f'\n✅ Phase 2 done — Best AUC: {best_auc:.4f}')

## 📈 Cell 14 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
p1_end = NUM_EPOCHS_P1  # vertical line between phases

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, key_t, key_v, title in zip(
    axes,
    ['train_loss', 'train_acc', 'train_acc'],
    ['val_loss',   'val_acc',   'val_auc'],
    ['Loss', 'Accuracy', 'AUC-ROC']
):
    if title != 'AUC-ROC':
        ax.plot(epochs, history[key_t], label='Train', linewidth=1.5)
    ax.plot(epochs, history[key_v], label='Val' if title != 'AUC-ROC' else 'Val AUC', linewidth=1.5, color='darkorange')
    ax.axvline(x=p1_end + 0.5, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('✅ Saved training_curves.png')

## 🧪 Cell 15 — Test Set Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
ckpt = torch.load(os.path.join(SAVE_DIR, 'best_densenet121.pth'), map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best model — epoch {ckpt["epoch"]} phase {ckpt["phase"]} (Val AUC={ckpt["val_auc"]:.4f})')

test_loss, test_acc, test_auc, all_probs, all_labels = evaluate(model, test_loader, criterion, desc='Test')
all_preds = all_probs.argmax(axis=1)

print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  TEST RESULTS')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Loss     : {test_loss:.4f}')
print(f'  Accuracy : {test_acc:.4f}')
print(f'  AUC-ROC  : {test_auc:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## 🔲 Cell 16 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.ylabel('True label', fontsize=12)
plt.xlabel('Predicted label', fontsize=12)
plt.title('Confusion Matrix — Test Set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('✅ Saved confusion_matrix.png')

## 📉 Cell 17 — ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
colors  = ['#E24B4A', '#378ADD', '#639922', '#BA7517']

plt.figure(figsize=(8, 6))
for i, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Test Set', fontsize=13)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'roc_curves.png'), dpi=150)
plt.show()
print('✅ Saved roc_curves.png')

## 🔥 Cell 18 — Grad-CAM Visualization

In [ ]:
import cv2
import torch

class GradCAM:
    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target = model.features.denseblock4
        target.register_forward_hook(self._save_act)
        target.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):  self.activations = o.detach()
    def _save_grad(self, m, gi, go): self.gradients = go[0].detach()

    def generate(self, tensor, class_idx=None):
        self.model.eval()
        out = self.model(tensor)
        if class_idx is None:
            class_idx = out.argmax(dim=1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = torch.relu((weights * self.activations).sum(dim=1)).squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


def show_gradcam(image_path, model, save_name='gradcam.png'):
    orig   = Image.open(image_path).convert('RGB')
    tensor = val_test_transform(orig).unsqueeze(0).to(device)
    tensor.requires_grad_(True)

    gradcam      = GradCAM(model)
    cam, pred_i  = gradcam.generate(tensor)

    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
    heatmap     = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap     = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    orig_arr    = np.array(orig.resize((IMAGE_SIZE, IMAGE_SIZE)))
    overlay     = (0.55 * orig_arr + 0.45 * heatmap).astype(np.uint8)

    # Confidence scores
    with torch.no_grad():
        probs = torch.softmax(model(val_test_transform(orig).unsqueeze(0).to(device)), dim=1)
        probs = probs.squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(orig_arr, cmap='bone');  axes[0].set_title('Original X-ray');           axes[0].axis('off')
    axes[1].imshow(cam_resized, cmap='jet'); axes[1].set_title('Grad-CAM heatmap');         axes[1].axis('off')
    axes[2].imshow(overlay);                axes[2].set_title(f'Prediction: {CLASS_NAMES[pred_i]}\n({probs[pred_i]*100:.1f}%)'); axes[2].axis('off')
    plt.tight_layout()
    save_path = os.path.join(LOG_DIR, save_name)
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Prediction : {CLASS_NAMES[pred_i]} ({probs[pred_i]*100:.1f}%)')
    for i, (n, p) in enumerate(zip(CLASS_NAMES, probs)):
        print(f'  {n:15s}: {p*100:.1f}%')


# ── Run on a random test image ────────────────────────────
sample_row  = test_df.sample(1).iloc[0]
sample_path = os.path.join(DATA_DIR, sample_row['Image Index'])
print(f'Running Grad-CAM on: {sample_row["Image Index"]}  (true: {CLASS_NAMES[sample_row["label"]]})')
show_gradcam(sample_path, model)

## 🔮 Cell 19 — Predict a Single Image

In [ ]:
# ── Change this path to any chest X-ray image ────────────
IMAGE_TO_PREDICT = os.path.join(DATA_DIR, test_df.iloc[0]['Image Index'])
# ─────────────────────────────────────────────────────────

model.eval()
orig   = Image.open(IMAGE_TO_PREDICT).convert('RGB')
tensor = val_test_transform(orig).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze().cpu().numpy()

pred_name = CLASS_NAMES[probs.argmax()]
print(f'Image     : {os.path.basename(IMAGE_TO_PREDICT)}')
print(f'Prediction: {pred_name} ({probs.max()*100:.1f}%)')
print()
for name, p in zip(CLASS_NAMES, probs):
    bar = '█' * int(p * 40)
    print(f'  {name:15s} {p*100:5.1f}%  {bar}')

# Show the image
plt.figure(figsize=(4, 4))
plt.imshow(orig, cmap='bone')
plt.title(f'Prediction: {pred_name}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

## ✅ Cell 20 — Summary


In [ ]:
print('━'*50)
print('  TRAINING COMPLETE — SUMMARY')
print('━'*50)
print(f'  Model      : DenseNet-121 (ImageNet pretrained)')
print(f'  Classes    : {CLASS_NAMES}')
print(f'  Best AUC   : {best_auc:.4f}')
print()
print('  Files saved to Google Drive:')
for fname in os.listdir(SAVE_DIR):
    print(f'    checkpoints/{fname}')
for fname in os.listdir(LOG_DIR):
    print(f'    logs/{fname}')
print('━'*50)